In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" 

import tensorflow as tf
print("Placas de vídeo visíveis:", tf.config.list_physical_devices('GPU')) #tive que fazer isso pq tava dando um erro enorme e nao tava rolando o fit

I0000 00:00:1779571406.328135  390742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779571407.048071  390742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Placas de vídeo visíveis: []


In [2]:
import pandas as pd
import tensorflow as tf
import sklearn
import scikeras
import gc

In [3]:
from scikeras.wrappers import KerasClassifier
from tensorflow.keras import backend as k
from tensorflow.keras.models import Sequential
from sklearn.model_selection import GridSearchCV #imports

In [4]:
X = pd.read_csv('entradas_breast.csv')
y = pd.read_csv('saidas_breast.csv') #imports e entradas de do csv

In [5]:
def criar_rede(optimizer, loss, kernel_initializer, activation, neurons):
    
    k.clear_session()
    rede_neural = Sequential([
        tf.keras.layers.InputLayer(shape = (30,)),
        tf.keras.layers.Dense(units = neurons, activation = activation, kernel_initializer = kernel_initializer),
        tf.keras.layers.Dropout(rate = 0.2),
        tf.keras.layers.Dense(units = neurons, activation = activation, kernel_initializer = kernel_initializer),
        tf.keras.layers.Dropout(rate = 0.2),
        tf.keras.layers.Dense(units = 1, activation = 'sigmoid') #funcao pra criar a rede neural com 2 camadas ocultas
    ])
    rede_neural.compile(optimizer = optimizer, loss = loss, metrics = ['binary_accuracy']) #compilacao dela

    gc.collect()
    
    return rede_neural #e volta ela

In [6]:
rede_neural = KerasClassifier(model = criar_rede)

In [7]:
parametros = {
    'batch_size' : [10, 30],
    'epochs': [50, 100],
    'model_optimizer': ['adam', 'sgd'],
    'model_loss': ['binary_crossentropy', 'hinge'],
    'model_kernel_initializer': ['random_uniform', 'normal'],
    'model_activation': ['relu', 'tanh'],
    'model_neurons': [16, 8]
} #parametros pra colocar na rede 

In [8]:
parametros

{'batch_size': [10, 30],
 'epochs': [50, 100],
 'model_optimizer': ['adam', 'sgd'],
 'model_loss': ['binary_crossentropy', 'hinge'],
 'model_kernel_initializer': ['random_uniform', 'normal'],
 'model_activation': ['relu', 'tanh'],
 'model_neurons': [16, 8]}

In [9]:
parametros = { #parametros modificados pra rodar 
    'batch_size' : [10, 30],
    'epochs': [50],
    'model__optimizer': ['adam'],
    'model__loss': ['binary_crossentropy'],
    'model__kernel_initializer': ['random_uniform', 'normal'],
    'model__activation': ['relu'],
    'model__neurons': [16]
}

In [10]:
grid_search = GridSearchCV(estimator = rede_neural, param_grid = parametros, scoring = 'accuracy', cv = 5) #definicoes da rede

In [11]:
grid_search = grid_search.fit(X, y) #treinando

Epoch 1/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 984us/step - binary_accuracy: 0.5319 - loss: 1.4694 
Epoch 2/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 862us/step - binary_accuracy: 0.6330 - loss: 0.6781
Epoch 3/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 936us/step - binary_accuracy: 0.6242 - loss: 0.5900
Epoch 4/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 793us/step - binary_accuracy: 0.6813 - loss: 0.5409
Epoch 5/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 992us/step - binary_accuracy: 0.6967 - loss: 0.5469
Epoch 6/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - binary_accuracy: 0.7319 - loss: 0.5227
Epoch 7/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 723us/step - binary_accuracy: 0.7429 - loss: 0.5093
Epoch 8/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 835us/step - binary_accuracy: 0.7297 - loss: 0.4876
Epoch 9/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 745us/step - binary_accuracy: 0.7736 - loss: 0.4683
Epoch 10/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 734us/step - binary_accuracy: 0.8044 - loss: 0.4472
Epoch 11/50
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 739us/step - binary_accura

In [12]:
melhores_param = grid_search.best_params_ #pegando os melhores parametros

In [13]:
print(melhores_param)

{'batch_size': 30, 'epochs': 50, 'model__activation': 'relu', 'model__kernel_initializer': 'normal', 'model__loss': 'binary_crossentropy', 'model__neurons': 16, 'model__optimizer': 'adam'}


In [14]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao) #melhor precisao

0.8698959788852662
